# 02 — Learn LangGraph (the weekly agent)

Notebook 01 built *one* LLM call. LangGraph wraps *many* steps into a stateful machine with
branching and a human in the loop.

**Concepts:**
1. `StateGraph` + a `TypedDict` state shared by every node.
2. `add_node` / `add_edge` / `add_conditional_edges` — the wiring.
3. `interrupt()` + a checkpointer — pause for human approval, then resume.

Our graph: `load_universe -> screen -> council -> rank_pick -> (draft_card -> human_approval -> write_record)`,
with branches to sit out a week or to stop if the human declines.

## See the graph

In [ ]:
from quantwheel.agents.graph import build_weekly_agent
agent = build_weekly_agent()              # llm=None -> live Claude council
print(agent.get_graph().draw_ascii())     # visualise the flow

## Human-in-the-loop: pause, then resume

`run_weekly_agent(auto_approve=None)` runs until the `interrupt()` in `human_approval` and
returns the paused state (look for `__interrupt__`). A UI shows the card, then calls
`resume_weekly_agent(True/False)` with the same `thread_id`. **No order is ever placed by code.**

In [ ]:
from quantwheel.agents.graph import run_weekly_agent, resume_weekly_agent

# Needs ANTHROPIC_API_KEY (council) + internet (scan).
paused = run_weekly_agent(["NIO", "F", "SOFI"], auto_approve=None, thread_id="demo")
if "__interrupt__" in paused:
    print("PAUSED for approval:", paused["__interrupt__"][0].value["summary"])
    final = resume_weekly_agent(True, thread_id="demo")   # approve
    print(final["message"])
else:
    print("Sat out:", paused.get("message"))

## One-shot (auto-approve)

For a cron/non-interactive run you can answer the gate inline.

In [ ]:
out = run_weekly_agent(["NIO"], auto_approve=True, thread_id="oneshot")
print(out["message"])
if out.get("card"):
    print(out["card"].exit_plan)

## Takeaways
- The graph makes branching (sit-out / decline) and the human gate *explicit and visual*.
- Swapping `MemorySaver` for `SqliteSaver` makes the pause survive a restart.
- The same `services/` functions power the dashboard — the graph is the *agentic* face of them.